In [ ]:
import sys
from pathlib import Path

print("Current dir:", Path.cwd())
sys.path.append(str(Path.cwd().parent))

import config as config
print("Loaded from:", config.__file__)

In [ ]:
from config import get_spark_session, s3_path

spark = get_spark_session("bronze-to-silver")

reviews = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .csv(
        s3_path(
            "bronze",
            "order_reviews",
            "olist_order_reviews_dataset.csv"
        )
    )
)

Data profiling, understand the data before transformation


In [ ]:
reviews.printSchema()

print(f"Number of records: {reviews.count()}")

reviews.show(10, truncate=False)

reviews.describe().show()

from pyspark.sql.functions import col, count, when

reviews.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in reviews.columns
]).show()

In [ ]:
from pyspark.sql.functions import col

reviews.groupBy("review_id") \
    .count() \
    .filter(col("count") > 1) \
    .show(truncate=False)

In [ ]:
total = reviews.count()

distinct = reviews.distinct().count()

print("Total Rows:", total)
print("Distinct Rows:", distinct)
print("Duplicate Rows:", total - distinct)

In [ ]:
reviews.filter(
    (col("review_score") < 1) |
    (col("review_score") > 5)
).show()

In [ ]:
reviews.filter(
    col("review_id") == "f144ac1998474653203c861be02fd31f"
).show(truncate=False)

In [ ]:
reviews.filter(
    col("review_answer_timestamp") <
    col("review_creation_date")
).show(truncate=False)